In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import torch
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

# Google Colab-এ ড্রাইভ মাউন্ট করতে (যদি প্রয়োজন হয়) নিচের লাইনগুলি আনকমেন্ট করুন
# from google.colab import drive
# drive.mount('/content/drive')

# GPU সাপোর্ট সেটআপ
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ডেটাসেট ও মডেল সংরক্ষণের পাথ
data_dir = '/content/drive/MyDrive/processed_data'
train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'test')
save_dir = '/content/drive/MyDrive/Model'
os.makedirs(save_dir, exist_ok=True)

# ট্রেনিং ও ভ্যালিডেশনের জন্য ইমেজ ট্রান্সফর্মেশন
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),                # AlexNet-এ 224x224 ইনপুট
    transforms.RandomHorizontalFlip(p=0.5),       # এলোমেলো হরাইজন্টাল ফ্লিপ
    transforms.RandomRotation(20),                # এলোমেলো ঘোরানো
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet গড়/স্ট্যান্ড.ডিভ.
                         std=[0.229, 0.224, 0.225])
])
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ImageFolder দিয়ে ডেটাসেট লোড করা
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform_train)
val_dataset   = datasets.ImageFolder(root=test_dir,  transform=transform_val)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4)
val_loader    = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4)

# শ্রেণীর সংখ্যা নির্ণয়
num_classes = len(train_dataset.classes)
print(f'Number of classes: {num_classes}')

# AlexNet মডেল (ড্রপআউট লেয়ার অন্তর্ভুক্ত) লোড করা
model = models.alexnet(pretrained=False)
# চূড়ান্ত লেয়ারকে আমাদের ক্লাস সংখ্যার সঙ্গে সামঞ্জস্যপূর্ণ করা
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
model = model.to(device)

# লস ফাংশন ও অপ্টিমাইজার (weight decay দিয়ে regularization:contentReference[oaicite:3]{index=3})
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)

# ঐচ্ছিক: লার্নিং রেট শিডিউলার
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# ট্রেনিং লুপ
num_epochs = 10
for epoch in range(num_epochs):
    model.train()  # ড্রপআউট সক্রিয় করা হয়
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)

    # ভ্যালিডেশন পর্যায়
    model.eval()   # ড্রপআউট ইত্যাদি নিষ্ক্রিয়
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_loss = val_loss / len(val_loader.dataset)
    val_acc = 100 * correct / total

    print(f'Epoch {epoch+1}/{num_epochs}: '
          f'Train Loss: {epoch_loss:.4f}, '
          f'Val Loss: {val_loss:.4f}, '
          f'Val Acc: {val_acc:.2f}%')

    scheduler.step()

# প্রশিক্ষণ শেষে মডেল .pth ফাইলে সংরক্ষণ
model_path = os.path.join(save_dir, 'alexnet_model.pth')
torch.save(model.state_dict(), model_path)
print(f'Model saved to {model_path}')


Using device: cpu


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Number of classes: 2
Epoch 1/10: Train Loss: 0.6932, Val Loss: 0.6923, Val Acc: 50.00%
Epoch 2/10: Train Loss: 0.6923, Val Loss: 0.6914, Val Acc: 50.00%
Epoch 3/10: Train Loss: 0.6909, Val Loss: 0.6897, Val Acc: 55.00%
Epoch 4/10: Train Loss: 0.6888, Val Loss: 0.6865, Val Acc: 62.50%
Epoch 5/10: Train Loss: 0.6836, Val Loss: 0.6784, Val Acc: 56.00%
Epoch 6/10: Train Loss: 0.6689, Val Loss: 0.6536, Val Acc: 62.75%
Epoch 7/10: Train Loss: 0.6206, Val Loss: 0.5859, Val Acc: 69.75%
Epoch 8/10: Train Loss: 0.5741, Val Loss: 0.5779, Val Acc: 70.00%
Epoch 9/10: Train Loss: 0.5637, Val Loss: 0.5748, Val Acc: 70.00%
Epoch 10/10: Train Loss: 0.5523, Val Loss: 0.5711, Val Acc: 70.25%
Model saved to /content/drive/MyDrive/Model/alexnet_model.pth
